# eval — compare runs

Loads every `runs/<dataset>/<timestamp>_<githash>/{config.json,summary.csv}` and plots metrics across runs.
Each run is one bar/point keyed by `dataset/timestamp`, so scores are reproducible. Run this notebook from `harness/` or `eval/`.

Two metric families:
- **Retrieval** (P@K, R@K, MRR, context recall/precision) — did the gold notes get retrieved.
- **Answer scoring** (accuracy + labels: CORRECT / INCORRECT_REFUSAL / WRONG_STATE / PARTIAL / …) — did the model *use* them. Only present for runs scored with the state-aware `metrics.py`; per-question detail lives in each run's `answers.csv`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# runs/ now lives under eval/ and is nested per dataset: runs/<dataset>/<run>/
# Works whether the notebook's cwd is eval/ or eval/harness/.
cwd = Path.cwd()
RUNS_DIR = (cwd / 'runs') if (cwd / 'runs').exists() else (cwd.parent / 'runs')

def load_runs(runs_dir: Path) -> pd.DataFrame:
    rows = []
    if not runs_dir.exists():
        print(f"no runs dir at {runs_dir.resolve()}")
        return pd.DataFrame(rows)
    for ds_dir in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        for run_dir in sorted(p for p in ds_dir.iterdir() if p.is_dir()):
            summary_path = run_dir / 'summary.csv'
            config_path = run_dir / 'config.json'
            if not summary_path.exists():
                continue
            s = pd.read_csv(summary_path)
            # Only metric/value run summaries belong here. Skip anything else
            # (e.g. an old failure_mode,count,percent breakdown).
            if list(s.columns) != ['metric', 'value']:
                print(f"skipping {ds_dir.name}/{run_dir.name}: unexpected columns {list(s.columns)}")
                continue
            summary = dict(zip(s['metric'], s['value']))
            config = json.loads(config_path.read_text()) if config_path.exists() else {}
            rows.append({
                'dataset': ds_dir.name,
                'run': run_dir.name,
                'label': f"{ds_dir.name}/{run_dir.name[:13]}",
                **config,
                **summary,
            })
    return pd.DataFrame(rows)

df = load_runs(RUNS_DIR)
df

In [ ]:
# Coerce metric columns to numeric and plot across runs.
metric_cols = [c for c in df.columns if c.startswith(('precision_at_', 'recall_at_')) or c in ('mrr', 'context_recall', 'context_precision')]
plot_df = df.set_index('label')[metric_cols].apply(pd.to_numeric, errors='coerce')

ax = plot_df.plot(kind='bar', figsize=(11, 5))
ax.set_title('Retrieval metrics by run (dataset/timestamp)')
ax.set_ylabel('score')
ax.set_ylim(0, 1)
ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

In [ ]:
# Gate view: context recall vs precision. High recall + low precision => reranker is the lever.
gate = df.set_index('label')[['context_recall', 'context_precision']].apply(pd.to_numeric, errors='coerce')
gate

## Answer scoring

Did the model *use* the retrieved notes? `answer_accuracy` = CORRECT / answerable. The labels split the failures: **INCORRECT_REFUSAL** (refused despite a gold note being retrieved), **WRONG_STATE** (named a competing state — e.g. the current job when *former* was asked), **PARTIAL** (multi-part answer, only some parts), plus CORRECT/FAILED refusals on the unanswerable set. The last cell breaks accuracy down by `trap_type` and `state_role` — that's where the current-vs-former-vs-prospective signal shows up.

In [ ]:
# Answer metrics from summary.csv (present only for runs scored with the state-aware metrics.py).
ans_metric_cols = ['answer_accuracy', 'n_correct', 'partial', 'incorrect_refusals',
                   'refusals_no_context', 'wrong_state', 'wrong_other', 'failed_refusals']
have = [c for c in ans_metric_cols if c in df.columns]
if not have:
    print('No answer-scoring columns in summary.csv -- re-run run_eval.py on a dataset.')
else:
    adf = df.set_index('label')[have].apply(pd.to_numeric, errors='coerce').dropna(how='all')
    display(adf)
    if 'answer_accuracy' in adf.columns:
        ax = adf['answer_accuracy'].dropna().plot(kind='bar', figsize=(10, 4), color='seagreen')
        ax.set_title('answer_accuracy by run (CORRECT / answerable)'); ax.set_ylim(0, 1); ax.set_ylabel('accuracy')
        plt.tight_layout(); plt.show()
    comp = [c for c in ['n_correct', 'partial', 'incorrect_refusals', 'refusals_no_context',
                        'wrong_state', 'wrong_other', 'failed_refusals'] if c in adf.columns]
    comp_df = adf[comp].dropna(how='all')
    if not comp_df.empty:
        ax = comp_df.plot(kind='bar', stacked=True, figsize=(11, 5), colormap='tab20')
        ax.set_title('answer label composition by run'); ax.set_ylabel('# questions')
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0)); plt.tight_layout(); plt.show()

In [ ]:
# Answer accuracy by trap_type / state_role -- the state-reasoning view (from each run's answers.csv).
def load_answers(runs_dir: Path) -> pd.DataFrame:
    frames = []
    if not runs_dir.exists():
        return pd.DataFrame()
    for ds_dir in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        for run_dir in sorted(p for p in ds_dir.iterdir() if p.is_dir()):
            ap = run_dir / 'answers.csv'
            if not ap.exists():
                continue
            a = pd.read_csv(ap)
            a['run_label'] = f"{ds_dir.name}/{run_dir.name[:13]}"
            frames.append(a)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

ans = load_answers(RUNS_DIR)
if ans.empty:
    print('No answers.csv found -- run run_eval.py (or metrics.py) on a dataset first.')
else:
    answerable = ans[ans['answerability'] == 'answerable'].copy()
    answerable['correct'] = (answerable['label'] == 'CORRECT').astype(int)
    for dim in ['trap_type', 'state_role']:
        piv = answerable.pivot_table(index='run_label', columns=dim, values='correct', aggfunc='mean')
        print(f"\naccuracy by {dim} (CORRECT rate):")
        display(piv.round(2))
        ax = piv.plot(kind='bar', figsize=(11, 4))
        ax.set_title(f'answer accuracy by {dim}'); ax.set_ylim(0, 1); ax.set_ylabel('accuracy')
        ax.legend(loc='upper left', bbox_to_anchor=(1.0, 1.0)); plt.tight_layout(); plt.show()